# Frontier Model Benchmark — GPT-5-mini on Granulometry

Before using a frontier model to generate training data (Task 4 SEAL approach),
verify it can actually classify these images correctly.

Tests GPT-5-mini (Azure OpenAI) on the same 108 test images, zero-shot and few-shot.
Compares against Qwen2.5-VL-3B baseline from the main benchmark.

## Setup

In [ ]:
import os, json, re, time, base64
import numpy as np
from PIL import Image
from collections import defaultdict, Counter
import matplotlib.pyplot as plt

# !pip install openai -q  # uncomment if not installed


## Config

In [ ]:
from openai import AzureOpenAI

TEST_DIR = '../../datasets/granulometry/test'
MANIFEST = '../../datasets/granulometry/test_manifest.json'
REF_IMAGE_PATH = 'examples_classification_data.png'

# Azure OpenAI — GPT-5-mini
AZURE_ENDPOINT = 'https://ether-project-resource.openai.azure.com/'
DEPLOYMENT = 'gpt-5-mini'
API_VERSION = '2024-12-01-preview'
API_KEY = os.environ.get('AZURE_OPENAI_API_KEY', '')  # set below or via env

# Uncomment and set your key:
# API_KEY = 'your-key-here'

client = AzureOpenAI(
    azure_endpoint=AZURE_ENDPOINT,
    api_key=API_KEY,
    api_version=API_VERSION,
)

print(f'Endpoint: {AZURE_ENDPOINT}')
print(f'Deployment: {DEPLOYMENT}')
print(f'API key set: {bool(API_KEY)}')


## Prompts

In [ ]:
PROMPT_ZS = '''This is a top-down photo of concrete aggregate. GSD = 8.0 px/mm.

Classify it on two axes:

MAX PARTICLE SIZE - estimate the largest stone, round to 8, 16, or 32 mm.
Reference: at 8.0 px/mm, a 8mm stone = ~64px wide, 16mm = ~128px, 32mm = ~256px.

GRADING - describes size distribution (not absolute size):
- COARSE: most particles similar size, close to max. Few small particles. Uniform texture.
- MEDIUM: moderate mix of large and small.
- FINE: wide range of sizes. Many small particles fill gaps between larger ones. Dense, packed.

Respond with ONLY a JSON object:
{"max_particle_size_mm": <8|16|32>, "grading": "<coarse|medium|fine>"}'''

PROMPT_FS_REF = '''First image: reference chart. 3x3 grid of concrete aggregate photos.
COLUMNS (left to right) = max particle size: 8mm | 16mm | 32mm
ROWS (top to bottom) = grading: A (coarse) | B (medium) | C (fine)

Grading = size DISTRIBUTION:
- COARSE (A): uniform texture, particles mostly same size near max.
- MEDIUM (B): moderate mix.
- FINE (C): dense packed texture, many small particles filling gaps between big ones.

Column 32mm: row A = big uniform stones, row C = big stones + lots of small ones filling gaps.'''

PROMPT_FS_QUERY = '''Classify this photo. GSD = 8.0 px/mm.
Compare to the reference grid.
Respond with ONLY JSON: {"max_particle_size_mm": <8|16|32>, "grading": "<coarse|medium|fine>"}'''


## Helpers

In [ ]:
def encode_image(path):
    with open(path, 'rb') as f:
        return base64.b64encode(f.read()).decode('utf-8')

def call_zs(image_b64, prompt):
    resp = client.chat.completions.create(
        model=DEPLOYMENT, max_completion_tokens=256,
        messages=[{'role': 'user', 'content': [
            {'type': 'image_url', 'image_url': {'url': f'data:image/jpeg;base64,{image_b64}'}},
            {'type': 'text', 'text': prompt},
        ]}],
    )
    return resp.choices[0].message.content

def call_fs(ref_b64, image_b64, ref_prompt, query_prompt):
    resp = client.chat.completions.create(
        model=DEPLOYMENT, max_completion_tokens=256,
        messages=[{'role': 'user', 'content': [
            {'type': 'image_url', 'image_url': {'url': f'data:image/png;base64,{ref_b64}'}},
            {'type': 'text', 'text': ref_prompt},
            {'type': 'image_url', 'image_url': {'url': f'data:image/jpeg;base64,{image_b64}'}},
            {'type': 'text', 'text': query_prompt},
        ]}],
    )
    return resp.choices[0].message.content

def parse_response(raw):
    if not raw: return None
    raw = re.sub(r'```json\s*', '', raw)
    raw = re.sub(r'```\s*', '', raw).strip()
    try:
        obj = json.loads(raw)
        if isinstance(obj, dict): return obj
    except: pass
    m = re.search(r'\{.*\}', raw, re.DOTALL)
    if m:
        try: return json.loads(m.group())
        except: pass
    # Flexible regex: handle quoted or unquoted numbers
    size_m = re.search(r'["\']?max_particle_size_mm["\']?\s*:\s*["\']?(\d+)["\']?', raw)
    grad_m = re.search(r'["\']?grading["\']?\s*:\s*["\']?(\w+)["\']?', raw)
    if size_m and grad_m:
        return {'max_particle_size_mm': int(size_m.group(1)), 'grading': grad_m.group(1)}
    return None

print('Helpers ready.')


## Load Data

In [ ]:
with open(MANIFEST) as f:
    manifest = json.load(f)
print(f'Test images: {len(manifest)}')

# Load reference image for few-shot
ref_b64 = None
if os.path.exists(REF_IMAGE_PATH):
    ref_b64 = encode_image(REF_IMAGE_PATH)
    print(f'Reference image loaded: {REF_IMAGE_PATH}')
else:
    print(f'WARNING: {REF_IMAGE_PATH} not found')


## Quick Test (3 images: A32, B16, C8)

In [ ]:
quick_classes = ['A32', 'B16', 'C8']
quick = [next(e for e in manifest if e['class'] == cls) for cls in quick_classes]

for mode in ['zero-shot', 'few-shot']:
    print(f'\n{"=" * 60}')
    print(f'Quick test — {mode}')
    print(f'{"=" * 60}')
    print(f'{"Class":<6} {"GT":>12} {"Predicted":>16} {"Size":>5} {"Grade":>6}')
    print('-' * 50)
    for entry in quick:
        img_path = os.path.join(TEST_DIR, entry['image'])
        img_b64 = encode_image(img_path)
        try:
            if mode == 'zero-shot':
                raw = call_zs(img_b64, PROMPT_ZS)
            else:
                raw = call_fs(ref_b64, img_b64, PROMPT_FS_REF, PROMPT_FS_QUERY)
        except Exception as e:
            print(f'{entry["class"]:<6} ERROR: {e}')
            continue
        parsed = parse_response(raw)
        gt_s = entry['max_particle_size_mm']; gt_g = entry['grading']
        if parsed:
            ps = parsed.get('max_particle_size_mm'); pg = parsed.get('grading', '').lower()
            if isinstance(ps, str): ps = int(ps) if ps.isdigit() else ps
            sv = '✓' if ps == gt_s else '✗'; gv = '✓' if pg == gt_g else '✗'
            print(f'{entry["class"]:<6} {gt_s}mm {gt_g:<8} {str(ps)}mm {pg:<10} {sv:>5} {gv:>6}')
        else:
            print(f'{entry["class"]:<6} {gt_s}mm {gt_g:<8} PARSE FAIL')
            print(f'  Raw: {raw[:200]}')
        time.sleep(0.3)


## Full Benchmark (108 images)

In [ ]:
def run_frontier_benchmark(manifest, mode, ref_b64=None):
    results = []; c_size = 0; c_grading = 0; v_json = 0; t_time = 0
    for i, entry in enumerate(manifest):
        img_path = os.path.join(TEST_DIR, entry['image'])
        if not os.path.exists(img_path): continue
        img_b64 = encode_image(img_path)
        t = time.time()
        try:
            if mode == 'zero-shot':
                raw = call_zs(img_b64, PROMPT_ZS)
            else:
                raw = call_fs(ref_b64, img_b64, PROMPT_FS_REF, PROMPT_FS_QUERY)
        except Exception as e:
            raw = f'ERROR: {e}'
        elapsed = time.time() - t; t_time += elapsed
        parsed = parse_response(raw)
        gt_s = entry['max_particle_size_mm']; gt_g = entry['grading']
        s_ok = False; g_ok = False
        if parsed:
            v_json += 1
            ps = parsed.get('max_particle_size_mm')
            if isinstance(ps, str): ps = int(ps) if ps.isdigit() else None
            if ps == gt_s: s_ok = True; c_size += 1
            if parsed.get('grading', '').lower() == gt_g: g_ok = True; c_grading += 1
        results.append({'image': entry['image'], 'class': entry['class'],
            'gt_size': gt_s, 'gt_grading': gt_g, 'predicted': parsed, 'raw': raw,
            'size_correct': s_ok, 'grading_correct': g_ok, 'valid_json': parsed is not None,
            'time_s': round(elapsed, 2)})
        if (i+1) % 20 == 0:
            n = i+1
            print(f'  [{n}/{len(manifest)}] Size: {c_size}/{n} ({c_size/n*100:.0f}%) | '
                  f'Grading: {c_grading}/{n} ({c_grading/n*100:.0f}%) | JSON: {v_json}/{n}')
        time.sleep(0.25)  # rate limit
    return results, c_size, c_grading, v_json, t_time

print('Benchmark function ready.')


### Run Zero-Shot (108 images)

In [ ]:
print('Running GPT-5-mini zero-shot...')
fr_zs_results, fr_zs_size, fr_zs_grading, fr_zs_json, fr_zs_time = run_frontier_benchmark(manifest, 'zero-shot')
print(f'Done. {len(fr_zs_results)} images in {fr_zs_time:.0f}s')


### Run Few-Shot (108 images)

In [ ]:
print('Running GPT-5-mini few-shot...')
fr_fs_results, fr_fs_size, fr_fs_grading, fr_fs_json, fr_fs_time = run_frontier_benchmark(manifest, 'few-shot', ref_b64=ref_b64)
print(f'Done. {len(fr_fs_results)} images in {fr_fs_time:.0f}s')


## Results: GPT-5-mini vs Qwen2.5-VL-3B

In [ ]:
def summarize(label, results, c_size, c_grading, v_json, t_time):
    n = len(results)
    both = sum(1 for r in results if r['size_correct'] and r['grading_correct'])
    return {'label': label, 'n': n, 'json': round(v_json/n*100,1), 'size': round(c_size/n*100,1),
            'grading': round(c_grading/n*100,1), 'both': round(both/n*100,1), 'time': round(t_time/n,2)}

fr_zs_s = summarize('GPT-5-mini (ZS)', fr_zs_results, fr_zs_size, fr_zs_grading, fr_zs_json, fr_zs_time)
fr_fs_s = summarize('GPT-5-mini (FS)', fr_fs_results, fr_fs_size, fr_fs_grading, fr_fs_json, fr_fs_time)

# Load Qwen baselines
rows = []
for mode in ['zero-shot', 'few-shot']:
    path = f'benchmark_results_{mode}.json'
    if os.path.exists(path):
        with open(path) as f:
            d = json.load(f)
        rows.append({'label': f'Qwen2.5-VL-3B ({mode})', 'json': d['json_validity_pct'],
            'size': d['size_accuracy_pct'], 'grading': d['grading_accuracy_pct'],
            'both': d['both_correct_pct'], 'time': d['avg_inference_time_s']})
rows.append(fr_zs_s)
rows.append(fr_fs_s)

print(f'{"Method":<28} {"JSON":>6} {"Size":>7} {"Grade":>7} {"Both":>7} {"Time":>6}')
print('-' * 63)
for r in rows:
    print(f'{r["label"]:<28} {r["json"]:>5.0f}% {r["size"]:>6.1f}% {r["grading"]:>6.1f}% {r["both"]:>6.1f}% {r["time"]:>5.1f}s')


## Per-Class Breakdown

In [ ]:
for label, results in [('GPT-5-mini ZS', fr_zs_results), ('GPT-5-mini FS', fr_fs_results)]:
    print(f'\n{label}:')
    print(f'{"Class":<6} {"Size":>6} {"Grade":>6} {"Both":>6}')
    print('-' * 28)
    by_class = defaultdict(list)
    for r in results: by_class[r['class']].append(r)
    for cls in sorted(by_class):
        cr = by_class[cls]
        sc = sum(1 for r in cr if r['size_correct'])
        gc = sum(1 for r in cr if r['grading_correct'])
        bc = sum(1 for r in cr if r['size_correct'] and r['grading_correct'])
        print(f'{cls:<6} {sc:>3}/{len(cr)} {gc:>3}/{len(cr)} {bc:>3}/{len(cr)}')


## Visualization

In [ ]:
methods = [r['label'] for r in rows]
size_vals = [r['size'] for r in rows]
grad_vals = [r['grading'] for r in rows]
both_vals = [r['both'] for r in rows]

x = np.arange(len(methods))
w = 0.25
fig, ax = plt.subplots(figsize=(12, 5))
b1 = ax.bar(x - w, size_vals, w, label='Size Acc', color='steelblue')
b2 = ax.bar(x, grad_vals, w, label='Grading Acc', color='coral')
b3 = ax.bar(x + w, both_vals, w, label='Both Correct', color='green')
ax.set_xticks(x); ax.set_xticklabels(methods, rotation=15, ha='right')
ax.set_ylabel('Accuracy %'); ax.set_ylim(0, 110)
ax.set_title('Qwen2.5-VL-3B vs GPT-5-mini — Granulometry Classification')
ax.legend()
ax.axhline(y=33.3, color='gray', linestyle='--', alpha=0.5, label='Random chance')
for bars in [b1, b2, b3]:
    for bar in bars:
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1, f'{bar.get_height():.0f}%', ha='center', fontsize=8)
plt.tight_layout(); plt.show()


## Save Results

In [ ]:
for label, s, results in [
    ('frontier-zero-shot', fr_zs_s, fr_zs_results),
    ('frontier-few-shot', fr_fs_s, fr_fs_results),
]:
    fname = f'benchmark_results_{label}.json'
    with open(fname, 'w') as f:
        json.dump({
            'model': f'{DEPLOYMENT} (Azure OpenAI)', 'mode': label, 'phase': 'frontier_baseline',
            'total_images': s['n'], 'json_validity_pct': s['json'],
            'size_accuracy_pct': s['size'], 'grading_accuracy_pct': s['grading'],
            'both_correct_pct': s['both'], 'avg_inference_time_s': s['time'],
            'results': results,
        }, f, indent=2)
    print(f'Saved {fname}')

print(f'\nConclusion:')
print(f'If GPT-5-mini >> Qwen baseline -> use it as teacher for SEAL augmentation in Task 4')
print(f'If GPT-5-mini ~= Qwen baseline -> task may need different approach')
